# DPO stage - DPO of SFT-GPT-2 small on hh-rlhf (`harmless-base`)

Takes the SFT anchor (`sft-gpt2-hh-21k/final`) and Direct-Preference-Optimizes it on the half
of `harmless-base` that SFT never saw. The output, `dpo-gpt2-hh-21k/final`, is the DPO'd side of
the SAE comparison.

The anchor is read, never written. It is the policy initialization and - through
`ref_model=None` with `precompute_ref_log_probs=True` - the frozen reference inside the loss.
Everything this notebook writes is a sibling directory of it.

**Run order:** mount Drive, set secrets, then run the cells top to bottom. The SFT run must have
finished and written `sft-gpt2-hh-21k/manifest.json`.

## 1. Drive

First, before anything else. Checkpoints go to Drive, not to the VM disk - a disconnect
partway through an unmounted run loses every checkpoint.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# trl pinned exactly: DPOConfig / DPOTrainer arguments and the dataset columns read in the step-0
# cell were checked against 1.13.0. A fresh VM on resume must not pick up a newer release mid-run.
!pip -q install -U transformers datasets accelerate wandb "trl==1.13.0"

## 2. Secrets and environment

The key comes from the Colab secrets panel (the key icon in the left sidebar), never from a
cell. A pasted key gets saved into the notebook's stored output; so does an interactive
`wandb.login()`. Add a secret named `WANDB_API_KEY` and give this notebook access to it.

In [ ]:
import os

from google.colab import userdata

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = "DPO-SAE"     # shared with the SFT and SAE runs
os.environ["WANDB_LOG_MODEL"] = "false"     # checkpoints belong on Drive, not in wandb
os.environ["WANDB_WATCH"] = "false"

# Optional: park the HF cache on Drive so harmless-base and the cached reference logprobs
# survive a fresh VM. Without it, a resume on a new VM re-runs the reference pass.
# os.environ["HF_HOME"] = "/content/drive/MyDrive/DPO-SAE/hf-cache"

## 3. Configuration

Three things here are load-bearing.

**The anchor is read-only.** Every output path is asserted to sit outside the SFT run directory,
and the anchor's `model.safetensors` is hashed before training and again after. The hash goes
into this run's manifest so the SAE stage can check that the reference, the initialization and
the SAE dictionary all refer to the same bytes.

**No pair counter.** `DPOTrainer` owns its collator, so the SFT run's counting machinery is not
ported. Evaluation and checkpointing run off `eval_steps` / `save_steps` in optimizer steps,
derived in section 5 once the dataset size is known. DPO and SFT checkpoints therefore do not
share an x-axis; the comparison that matters is endpoint to endpoint.

**Fresh runs and resumes are guarded.** A fresh run refuses to start over an existing
`_resume/` checkpoint or an existing `final/` - the latter may already be consumed downstream.

In [ ]:
import hashlib
import json
import math
import os

import datasets
import torch
import transformers
import trl
from transformers.trainer_utils import get_last_checkpoint

DATASET = "Anthropic/hh-rlhf"
DATA_DIR = "harmless-base"

DRIVE_ROOT = "/content/drive/MyDrive/DPO-SAE"
SFT_RUN_DIR = f"{DRIVE_ROOT}/sft-gpt2-hh-21k"
ANCHOR = f"{SFT_RUN_DIR}/final"                 # read-only: policy init, pi_ref, SAE anchor
SFT_MANIFEST = f"{SFT_RUN_DIR}/manifest.json"

RUN_NAME = "dpo-gpt2-hh-21k"
RUN_DIR = f"{DRIVE_ROOT}/{RUN_NAME}"
CKPT_DIR = f"{RUN_DIR}/checkpoints"      # kept: weights + tokenizer at every save
FINAL_DIR = f"{RUN_DIR}/final"           # the DPO'd model the SAE analysis consumes
RESUME_DIR = f"{RUN_DIR}/_resume"        # rolling Trainer state, disconnect insurance only

RESUME = False                # flip to True after a disconnect, then re-run from the top

MAX_LEN = 512                 # prompt + completion, per sequence
MIN_COMPLETION_TOKENS = 16    # same rule as SFT: the prompt must leave room for a real completion
VAL_PAIRS = 512               # fixed val slice off the front of the reserved half

BETA = 0.1
LR = 5e-6
WARMUP_RATIO = 0.03           # turned into warmup_steps; transformers 5.x dropped warmup_ratio
EPOCHS = 1
PER_DEVICE_BS = 4             # the real activation batch is 8: chosen and rejected both go forward
GRAD_ACCUM = 4                # 16 pairs per optimizer step
EVAL_BS = 4
PRECOMPUTE_REF_BS = None      # None = train/eval batch size; lower on its own if the ref pass OOMs
N_EVALS = 6                   # eval_steps = save_steps = total_steps // N_EVALS
LOG_EVERY_STEPS = 10
N_SAMPLE_GENERATIONS = 3


def inside(path, root):
    path, root = os.path.realpath(path), os.path.realpath(root)
    return os.path.commonpath([path, root]) == root


for d in (RUN_DIR, CKPT_DIR, FINAL_DIR, RESUME_DIR):
    assert not inside(d, SFT_RUN_DIR), f"{d} would write into the SFT run"
assert os.path.isfile(f"{ANCHOR}/model.safetensors"), f"no anchor weights under {ANCHOR}"

if os.path.exists(f"{FINAL_DIR}/model.safetensors"):
    raise RuntimeError(f"{FINAL_DIR} already holds a model - move it aside by hand, "
                       "it may already be consumed downstream")
existing = get_last_checkpoint(RESUME_DIR) if os.path.isdir(RESUME_DIR) else None
if existing and not RESUME:
    raise RuntimeError(f"{existing} exists - set RESUME=True to continue it, "
                       f"or clear {RESUME_DIR} to start over")
if RESUME and not existing:
    raise RuntimeError(f"RESUME=True but there is no checkpoint under {RESUME_DIR}")

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESUME_DIR, exist_ok=True)

os.environ["WANDB_RUN_ID"] = RUN_NAME
os.environ["WANDB_RESUME"] = "allow"

# DPOConfig turns bf16 on whenever fp16 is unset, so both are always passed explicitly.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMPERE = DEVICE == "cuda" and torch.cuda.get_device_capability()[0] >= 8
FP16 = DEVICE == "cuda" and not AMPERE     # T4 is Turing: fp16
BF16 = AMPERE                              # 3050 / A100: bf16

print(f"[env] device={DEVICE} fp16={FP16} bf16={BF16} run={RUN_NAME} resume={RESUME}")
print(f"[env] trl {trl.__version__} | transformers {transformers.__version__} | "
      f"datasets {datasets.__version__} | torch {torch.__version__}")
print(f"[env] anchor (read-only) -> {ANCHOR}")
print(f"[env] checkpoints -> {CKPT_DIR}")

In [ ]:
def sha256_file(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


ANCHOR_WEIGHTS = f"{ANCHOR}/model.safetensors"
ANCHOR_SHA256 = sha256_file(ANCHOR_WEIGHTS)
print(f"[anchor] model.safetensors sha256={ANCHOR_SHA256}")

# The split is read from the SFT manifest, never retyped: a typo silently reintroduces
# overlap with rows the reference has already memorized.
with open(SFT_MANIFEST) as fh:
    sft_manifest = json.load(fh)

assert sft_manifest["dataset"] == f"{DATASET}:{DATA_DIR}", sft_manifest["dataset"]
DATA_SEED = sft_manifest["data_seed"]
SFT_ROWS = tuple(sft_manifest["sft_rows"])
DPO_ROWS = tuple(sft_manifest["dpo_reserved_rows"])
assert DPO_ROWS[0] >= SFT_ROWS[1] or DPO_ROWS[1] <= SFT_ROWS[0], "manifest row ranges overlap"

print(f"[manifest] data_seed={DATA_SEED} sft_rows={SFT_ROWS} dpo_reserved_rows={DPO_ROWS}")

## 4. Preprocessing

**Rows.** `shuffle(seed=...)` is deterministic, so the manifest's seed reproduces SFT's exact
ordering; SFT took one range of it and this takes the other. The two cannot share a row by
construction - provided the hub copy still has the row count SFT saw, which is asserted.

**Splitting.** Same splitter as SFT, cut at the **last** `"\n\nAssistant:"`, but `chosen` and
`rejected` stay separate: TRL's explicit-prompt format wants `{prompt, chosen, rejected}` as plain
strings, and the prompt boundary is what lets the trainer sum logprobs over completion tokens
only.

Rows are dropped when the marker is absent, when `rejected` does not share the prefix (or has its
final assistant turn somewhere else), when either completion is empty, when the two completions
are identical, or when the prompt leaves fewer than `MIN_COMPLETION_TOKENS` inside `MAX_LEN`. The
prefix check is correctness, not hygiene - the loss assumes both completions answer the same
prompt.

The prompt-length rule is stricter than TRL's own (prompt shorter than `max_length`), so TRL's
filter drops nothing and the step count computed here is exact.

In [ ]:
MARKER = "\n\nAssistant:"


def split_dialogue(example):
    r"""Split an hh-rlhf row into {prompt, chosen, rejected} at the final turn marker.

    Rows look like:
        "\n\nHuman: how do I ...\n\nAssistant: well ...\n\nHuman: ok\n\nAssistant: sure"

    Unusable rows come back empty rather than raising, and keep_row drops them.
    """
    chosen, rejected = example["chosen"], example["rejected"]

    idx = chosen.rfind(MARKER)
    if idx == -1:
        return {"prompt": "", "chosen": "", "rejected": ""}

    prompt = chosen[: idx + len(MARKER)]

    # Guard: both completions must answer the same prompt - shared prefix, and rejected's
    # final assistant turn starting at the same offset rather than a turn later.
    if not rejected.startswith(prompt) or rejected.rfind(MARKER) != idx:
        return {"prompt": "", "chosen": "", "rejected": ""}

    return {
        "prompt": prompt,
        "chosen": chosen[idx + len(MARKER):],
        "rejected": rejected[idx + len(MARKER):],
    }


def keep_row(example):
    chosen, rejected = example["chosen"].strip(), example["rejected"].strip()
    return len(example["prompt"]) > 0 and len(chosen) > 0 and len(rejected) > 0 and chosen != rejected

In [ ]:
from datasets import load_dataset

raw = load_dataset(DATASET, data_dir=DATA_DIR, split="train")
RAW_N = len(raw)

# The shuffle only reproduces SFT's ordering over the same rows. A different row count means
# the ordering changed and the manifest's range no longer names SFT's complement.
assert RAW_N == sft_manifest["raw_rows"], (
    f"{DATA_DIR} train has {RAW_N} rows, SFT saw {sft_manifest['raw_rows']} - split not reproducible")

shuffled = raw.shuffle(seed=DATA_SEED)
dpo_raw = shuffled.select(range(*DPO_ROWS))

print(f"[data] {DATA_DIR} train: {RAW_N} rows")
print(f"[data] shuffle(seed={DATA_SEED}) -> DPO rows {DPO_ROWS}, SFT trained on {SFT_ROWS}")
print(f"[data] sampled {len(dpo_raw)} pairs for this stage")

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(ANCHOR)      # the anchor's own tokenizer, untouched
assert tok.pad_token == tok.eos_token, "anchor tokenizer should carry pad_token = eos_token"

split = dpo_raw.map(split_dialogue, remove_columns=dpo_raw.column_names)
after_split = split.filter(keep_row)
print(f"[data] marker / prefix / empty / identical filter: {len(split)} -> {len(after_split)} "
      f"({100 * len(after_split) / len(split):.1f}%)")


def count_prompt_tokens(batch):
    ids = tok(batch["prompt"], add_special_tokens=False)["input_ids"]
    return {"prompt_tokens": [len(x) for x in ids]}


counted = after_split.map(count_prompt_tokens, batched=True)
usable = counted.filter(lambda x: x["prompt_tokens"] <= MAX_LEN - MIN_COMPLETION_TOKENS)
usable = usable.remove_columns("prompt_tokens")
print(f"[data] prompt-length filter: {len(counted)} -> {len(usable)} "
      f"({100 * len(usable) / len(counted):.1f}%)")
print(f"[data] usable yield overall: {len(usable)}/{len(dpo_raw)} "
      f"({100 * len(usable) / len(dpo_raw):.1f}%)")

In [ ]:
# Fixed val slice off the front of the reserved half. The order is the seeded shuffle, so the
# same pairs land in val on every run.
dpo_val_ds = usable.select(range(VAL_PAIRS))
dpo_train_ds = usable.select(range(VAL_PAIRS, len(usable)))
TRAIN_PAIRS = len(dpo_train_ds)

VAL_PROMPTS = dpo_val_ds[:N_SAMPLE_GENERATIONS]["prompt"]

print(f"[data] train {TRAIN_PAIRS} pairs | val {len(dpo_val_ds)} pairs (held out, fixed)")
print(f"[data] columns: {dpo_train_ds.column_names}")

ex = dpo_train_ds[0]
print(f"\n  prompt   ...{ex['prompt'][-120:]!r}")
print(f"  chosen   {ex['chosen'][:120]!r}")
print(f"  rejected {ex['rejected'][:120]!r}")

## 5. DPOConfig

Values follow `CLAUDE.md`; the reasoning for each lives there. The ones worth repeating here:

- **`beta=0.1`, `learning_rate=5e-6`.** If `rewards/margins` spikes early, lower β first, the LR
  second.
- **`precompute_ref_log_probs=True`.** The reference pass runs once over train and val inside
  `DPOTrainer.__init__`, before any weight moves, and is cached next to the dataset. If that pass
  OOMs while training fits, lower `PRECOMPUTE_REF_BS` alone.
- **Left at default on purpose:** `loss_type` (sigmoid), `disable_dropout=True`, `sync_ref_model`,
  `f_divergence_type`, `use_weighting`, `label_smoothing`, `ld_alpha`. The first two are asserted
  after construction, so a TRL release that changes a default fails loudly instead of silently
  changing the algorithm.

**Saves.** `_resume/` holds full Trainer state (weights + optimizer + scheduler + RNG, ~1.5GB
each) and rolls with `save_total_limit=1`. The kept time series is the weights-only export into
`checkpoints/` at every save (next section) - about 0.5GB each, never rotated.

In [ ]:
from trl import DPOConfig

PAIRS_PER_STEP = PER_DEVICE_BS * GRAD_ACCUM
TOTAL_STEPS = math.ceil(TRAIN_PAIRS / PAIRS_PER_STEP) * EPOCHS
WARMUP_STEPS = math.ceil(WARMUP_RATIO * TOTAL_STEPS)
EVAL_STEPS = max(1, TOTAL_STEPS // N_EVALS)

cfg = DPOConfig(
    output_dir=RESUME_DIR,
    # --- DPO-specific ---
    beta=BETA,
    precompute_ref_log_probs=True,
    precompute_ref_batch_size=PRECOMPUTE_REF_BS,
    max_length=MAX_LEN,
    truncation_mode="keep_start",
    # --- optimization ---
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    num_train_epochs=EPOCHS,
    weight_decay=0.0,
    max_grad_norm=1.0,
    # --- batching / memory ---
    per_device_train_batch_size=PER_DEVICE_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    per_device_eval_batch_size=EVAL_BS,
    gradient_checkpointing=True,
    fp16=FP16,
    bf16=BF16,
    # --- cadence ---
    logging_steps=LOG_EVERY_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=EVAL_STEPS,
    save_total_limit=1,               # rolling resume state; the kept series is CKPT_DIR
    # --- bookkeeping ---
    report_to=["wandb"],
    run_name=RUN_NAME,
    seed=DATA_SEED,
)

assert cfg.loss_type in ("sigmoid", ["sigmoid"]), f"loss_type default changed: {cfg.loss_type}"
assert cfg.disable_dropout, "dropout would corrupt the cached-reference logprob ratio"
assert not cfg.sync_ref_model

print(f"[cfg] {TRAIN_PAIRS} pairs, {PAIRS_PER_STEP} pairs/step -> {TOTAL_STEPS} steps, "
      f"{WARMUP_STEPS} warmup")
print(f"[cfg] eval + save every {EVAL_STEPS} steps -> {TOTAL_STEPS // EVAL_STEPS} checkpoints + final")

## 6. Checkpoint exports

Two save mechanisms, two directories, same split as the SFT run:

- `RESUME_DIR` - Trainer's own `save_steps` checkpoints: full state, rolling, only for
  `resume_from_checkpoint` after a disconnect.
- `CKPT_DIR/step-NNNNNN` - written by the callback below whenever Trainer saves: `save_model()`
  plus the tokenizer, no optimizer state, loadable exactly like the anchor. These are the
  intermediates the feature analysis may read.

The callback also prints the headline eval metrics and a few sampled generations at each save.
Rewards and loss can move while the text degrades; the generations are the cheap check.

In [ ]:
import wandb
from transformers import TrainerCallback


@torch.no_grad()
def sample_generations(model, prompts, max_new_tokens=64):
    """Generate one completion per prompt, batch of one to sidestep padding side.

    Over-long prompts are cut from the left so the final "Assistant:" marker survives.
    """
    was_training = model.training
    model.eval()
    out = []
    for prompt in prompts:
        ids = tok(prompt, return_tensors="pt")["input_ids"][:, -(MAX_LEN - max_new_tokens):]
        ids = ids.to(model.device)
        gen = model.generate(
            input_ids=ids,
            attention_mask=torch.ones_like(ids),
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.9,
            temperature=0.8,
            pad_token_id=tok.eos_token_id,
        )
        out.append(tok.decode(gen[0, ids.shape[1]:], skip_special_tokens=True))
    model.train(was_training)
    return out


WATCH = ("rewards/accuracies", "rewards/margins", "rewards/chosen", "rewards/rejected",
         "logps/chosen", "logps/rejected", "loss")


class ExportCheckpoints(TrainerCallback):
    """Mirrors every Trainer save into CKPT_DIR as a weights + tokenizer export."""

    def __init__(self, ckpt_dir, prompts):
        self.ckpt_dir = ckpt_dir
        self.prompts = prompts
        self.trainer = None

    def on_train_begin(self, args, state, control, **kwargs):
        if wandb.run is not None:
            wandb.config.update(
                {
                    "stage": "dpo",
                    "anchor": ANCHOR,
                    "anchor_sha256": ANCHOR_SHA256,
                    "data_seed": DATA_SEED,
                    "sft_rows": list(SFT_ROWS),
                    "dpo_rows": list(DPO_ROWS),
                    "usable_pairs": len(usable),
                    "train_pairs": TRAIN_PAIRS,
                    "val_pairs": len(dpo_val_ds),
                    "max_len": MAX_LEN,
                },
                allow_val_change=True,
            )

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics:
            shown = "  ".join(f"{k}={metrics[f'eval_{k}']:.4f}" for k in WATCH
                              if f"eval_{k}" in metrics)
            print(f"[eval] step={state.global_step}  {shown}")

    def on_save(self, args, state, control, **kwargs):
        step = state.global_step
        path = f"{self.ckpt_dir}/step-{step:06d}"
        self.trainer.save_model(path)
        tok.save_pretrained(path)
        print(f"[ckpt] step={step} -> {path}")

        samples = sample_generations(self.trainer.model, self.prompts)
        for prompt, text in zip(self.prompts, samples):
            print(f"  prompt ...{prompt[-90:]!r}")
            print(f"  gen    {text[:200]!r}\n")
        if wandb.run is not None:
            table = wandb.Table(columns=["step", "prompt", "generation"])
            for prompt, text in zip(self.prompts, samples):
                table.add_data(step, prompt[-400:], text)
            wandb.log({"samples": table, "train/global_step": step})

## 7. Trainer

**`ref_model` is left unset, and that is correct.** With `ref_model=None` and
`precompute_ref_log_probs=True`, TRL keeps no second model: it runs the reference pass with the
policy itself inside `__init__`, while the policy still *is* the anchor, caches the logprobs as
dataset columns, and trains against those. That is why constructing the trainer takes a few
minutes.

**`data_collator` is not passed.** `DPOTrainer` builds its own preference collator for the
chosen/rejected concatenation, prompt masking and padding.

**On resume** the model is still loaded from `ANCHOR`, never from a `_resume/` checkpoint: the
reference pass above must see anchor weights. `train(resume_from_checkpoint=...)` swaps in the
trained weights and optimizer state afterwards.

In [ ]:
from transformers import AutoModelForCausalLM
from trl import DPOTrainer

# fp32 master weights: fp16 autocast with fp16 parameters cannot unscale gradients.
model = AutoModelForCausalLM.from_pretrained(ANCHOR, dtype=torch.float32)

exporter = ExportCheckpoints(CKPT_DIR, VAL_PROMPTS)

kwargs = dict(
    model=model,                   # the SFT anchor
    args=cfg,
    train_dataset=dpo_train_ds,    # {prompt, chosen, rejected}, reserved half only
    eval_dataset=dpo_val_ds,       # fixed 512-pair slice off the reserved half
    callbacks=[exporter],
)
try:                               # TRL renamed this around 0.12
    trainer = DPOTrainer(**kwargs, processing_class=tok)
except TypeError as err:
    if "processing_class" not in str(err):
        raise                      # a real error from inside __init__, not the rename
    trainer = DPOTrainer(**kwargs, tokenizer=tok)
exporter.trainer = trainer

assert trainer.ref_model is None, "a second reference model was loaded"
assert len(trainer.train_dataset) == TRAIN_PAIRS, (
    f"TRL dropped rows ({len(trainer.train_dataset)} vs {TRAIN_PAIRS}) - step math is off")
print(f"[train] reference logprobs cached for {len(trainer.train_dataset)} train "
      f"and {len(trainer.eval_dataset)} val pairs")

### Step-0 sanity check

At step 0 the policy *is* the reference, so `rewards/*` are zero by construction - they say
nothing yet about the anchor. What does say something is the reference logprob itself: the
per-token log-likelihood the anchor assigns to these completions, read straight from the cache.
The rows are unseen but in-distribution, so the perplexity on `chosen` should be in the same
ballpark as SFT's validation perplexity (not equal: SFT's was over prompt + completion). Far above
it means the anchor is off-distribution for this data.

The baseline evaluation then checks the other half of the premise: `eval_rewards/*` should be
~0 (up to fp16-vs-fp32 noise, since the cached pass ran without autocast). Clearly non-zero
means the cached reference is not the model being trained. **Skip this cell when resuming** -
it would log a second step-0 point into the resumed wandb run.

In [ ]:
def ref_logprob_per_token(ds, side):
    """Mean per-token reference logprob over completion tokens that survive truncation."""
    cols = ds.select_columns(["prompt_ids", f"{side}_ids", f"ref_{side}_logps"])[:]
    total_lp, total_tokens = 0.0, 0
    for p, c, lp in zip(cols["prompt_ids"], cols[f"{side}_ids"], cols[f"ref_{side}_logps"]):
        total_tokens += min(len(c), MAX_LEN - len(p))
        total_lp += lp
    return total_lp / total_tokens


for side in ("chosen", "rejected"):
    lp = ref_logprob_per_token(trainer.eval_dataset, side)
    print(f"[step0] anchor on val {side:8s}: {lp:.3f} nats/token, ppl {math.exp(-lp):.1f}")

baseline = trainer.evaluate()
for k in ("rewards/chosen", "rewards/rejected", "rewards/margins"):
    v = baseline[f"eval_{k}"]
    flag = "" if abs(v) < 0.05 else "   <-- expected ~0, check the reference cache"
    print(f"[step0] eval_{k} = {v:+.4f}{flag}")

## 8. Train

`report_to=["wandb"]` lets Trainer own `wandb.init` - calling it by hand as well splits the run in
two. After a disconnect: set `RESUME = True` in section 3 and re-run from the top.

In [ ]:
resume_from = get_last_checkpoint(RESUME_DIR) if RESUME else None
if RESUME:
    print(f"[train] resuming from {resume_from}")

trainer.train(resume_from_checkpoint=resume_from)

## 9. Final model and manifest

`FINAL_DIR` ends up structurally identical to the anchor directory and is loaded the same way.
The manifest records the anchor hash (re-checked here, after training), the row range, the
hyperparameters that define the algorithm, and the final validation metrics.

In [ ]:
trainer.save_model(FINAL_DIR)
tok.save_pretrained(FINAL_DIR)     # explicit, whatever this version's save_model does

final_metrics = trainer.evaluate()

assert sha256_file(ANCHOR_WEIGHTS) == ANCHOR_SHA256, "ANCHOR WEIGHTS CHANGED DURING THE RUN"

manifest = {
    "stage": "dpo",
    "anchor": ANCHOR,
    "anchor_sha256": ANCHOR_SHA256,
    "dataset": f"{DATASET}:{DATA_DIR}",
    "split": "train",
    "sampling": "ds.shuffle(seed=data_seed).select(range(*dpo_rows))",
    "data_seed": DATA_SEED,
    "raw_rows": RAW_N,
    "sft_rows": list(SFT_ROWS),
    "dpo_rows": list(DPO_ROWS),
    "usable_pairs": len(usable),
    "val_pairs": len(dpo_val_ds),
    "train_pairs": TRAIN_PAIRS,
    "max_len": MAX_LEN,
    "min_completion_tokens": MIN_COMPLETION_TOKENS,
    "beta": BETA,
    "loss_type": cfg.loss_type,
    "learning_rate": LR,
    "pairs_per_step": PAIRS_PER_STEP,
    "global_steps": trainer.state.global_step,
    "eval_save_steps": EVAL_STEPS,
    "checkpoints": sorted(os.listdir(CKPT_DIR)),
    "final_eval": final_metrics,
    "versions": {
        "trl": trl.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        "torch": torch.__version__,
    },
    "wandb": {"project": "DPO-SAE", "run_id": RUN_NAME},
}

with open(f"{RUN_DIR}/manifest.json", "w") as fh:
    json.dump(manifest, fh, indent=2)

print(json.dumps(manifest, indent=2))
print(f"\n[done] final model -> {FINAL_DIR}")

In [ ]:
# Final sanity check: still a coherent assistant turn after DPO?
for prompt, text in zip(VAL_PROMPTS, sample_generations(trainer.model, VAL_PROMPTS)):
    print(f"prompt ...{prompt[-120:]!r}")
    print(f"gen    {text!r}\n")

## 10. Before calling the run good

Loss alone will not tell you whether DPO worked. On the eval curves:

- **`rewards/accuracies`** should climb clearly above 0.5. Flat at 0.5 means no preference is
  being learned.
- **`rewards/margins`** should grow steadily. A sharp early spike means β or the LR is too high
  and the policy is running away from the reference - lower β first.
- **`rewards/chosen` vs `rewards/rejected`** should diverge. Both drifting negative with
  `rejected` falling faster is normal for DPO; `chosen` collapsing hard means the policy is
  lowering likelihood of everything.

Also:

- **Read the generations.** Rewards can rise while the text degrades into repetition or empty
  turns.
- **Check the step-0 cell.** Anchor perplexity near SFT's final val perplexity, `eval_rewards/*`
  ~0.
- **Watch Drive quota.** Six weights-only exports plus the final are ~3.5GB; `_resume/` adds one
  rolling ~1.5GB state, which can be deleted once `final/` and the manifest are written.
- **`final/` is now load-bearing.** Once the SAE analysis consumes it, never overwrite it - the
  same rule as the anchor.